In [1]:
import torch
import numpy as np
import random
from minicons import scorer

MODEL = "Qwen/Qwen3-VL-2B-Instruct"
CACHE_DIR = "/mnt/dv/wid/projects3/Rogers-muri-human-ai/zstuddiford"
SEED = 0

# --- seed everything ---
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- load model ---
lm = scorer.VLMScorer(MODEL, device="cuda", torch_dtype=torch.bfloat16, cache_dir=CACHE_DIR)

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [5]:
import random, pandas as pd, re
random.seed(0)

_SUBJ_SG = [
 'duck','dog','cat','bird','horse','fox','lion','frog','goat','bear','wolf','owl','seal','cow','pig',
 'hen','rat','crab','snail','moth','bee','ant','toad','hawk','crow','mole','newt','wasp','goose','mouse',
 'calf','lamb','colt','chick','cub','pup','kit','doe','ewe','ram','bull','mare','sow','drake','finch',
 'wren','lark','dove','gull','swan','heron','robin','sparrow','beetle','spider','lizard','otter','badger',
 'rabbit','hare','stoat','vole','shrew','weasel','ferret','mink','marten','raccoon','skunk','beaver',
 'squirrel','chipmunk','gopher','hamster','gerbil','hedgehog','porcupine','possum','koala','wombat','camel',
 'llama','donkey','mule','pony','zebra','antelope','gazelle','buffalo','bison','elk','panda','tiger',
 'leopard','cheetah','jaguar','cougar','lynx','bobcat','jackal','coyote','dingo','hyena','meerkat',
 'parrot','falcon','eagle','osprey','magpie','starling','pigeon','quail','pheasant','turkey','peacock',
 'flamingo','pelican','puffin','penguin','turtle','tortoise','gecko','iguana','salamander','minnow',
 'mantis','cricket','locust','hornet','beetle','weevil','aphid','earwig','firefly','ladybug',
 'dragonfly','grasshopper','caterpillar','centipede','millipede','tick','flea','gnat','midge','termite',
 'clam','oyster','mussel','prawn','shrimp','lobster','barnacle','urchin','jellyfish','starfish',
 'eel','trout','salmon','carp','perch','pike','bass','cod','herring','sardine',
 'shark','dolphin','whale','walrus','manatee','seahorse','stingray','octopus','squid','cuttlefish',
 'cobra','viper','python','adder','mamba','boa','rattler','skink','chameleon','monitor',
 'crane','stork','ibis','egret','plover','sandpiper','warbler','thrush','finch','bunting',
 'kestrel','harrier','buzzard','vulture','condor','raven','jackdaw','rook','jay','nuthatch',
 'mongoose','aardvark','pangolin','tapir','okapi','gnu','ibex','chamois','markhor','tahr',
 'wallaby','wombat','quokka','bandicoot','numbat','dunnart','quoll','potoroo','bilby','bettong',
]
SUBJECTS = [{'sg':w, 'pl':w+'s'} for w in dict.fromkeys(_SUBJ_SG)]  # dedup, regular +s plural
SUBJECTS = [s for s in SUBJECTS if s['sg'] != s['pl']]

_DIST_SG = [
 'tree','car','box','wall','table','house','pond','bridge','gate','fence','rock','lamp','chair','shed',
 'barn','cart','bench','post','window','door','crate','basket','barrel','wagon','ladder','pillar','statue',
 'hedge','well','stove','cabinet','desk','sofa','stool','mirror','clock','vase','plant','bush','stump',
 'log','boulder','fountain','column','arch','tower','cabin','tent','cottage','garage','porch','pole','sign',
 'mailbox','trough','crib','kennel','coop','hive','nest','burrow',
 'lantern','bucket','trellis','planter','urn','birdbath','sundial','gazebo','pergola','archway',
 'wheelbarrow','toolshed','greenhouse','beehive','scarecrow','flagpole','signpost','milestone','culvert','drainpipe',
 'cistern','aqueduct','footbridge','stile','turnstile','railing','banister','balustrade','parapet','buttress',
 'chimney','rooftop','awning','canopy','veranda','balcony','staircase','doorway','threshold','alcove',
]
DISTRACTORS = [{'sg':w, 'pl':(w+'es' if w.endswith(('x','s','sh','ch')) else w+'s')} for w in dict.fromkeys(_DIST_SG)]

PREPS = ["near the", "behind the", "beside the", "past the",
         "under the", "by the", "above the", "below the",
         "beneath the", "around the", "between the", "atop the",
         "outside the", "opposite the", "alongside the"]

_VERB = [
 ('jumps','jump'),('runs','run'),('eats','eat'),('sleeps','sleep'),('sings','sing'),('swims','swim'),
 ('dances','dance'),('plays','play'),('works','work'),('reads','read'),('waits','wait'),('rests','rest'),
 ('hides','hide'),('climbs','climb'),('digs','dig'),('hunts','hunt'),('feeds','feed'),('wanders','wander'),
 ('gathers','gather'),('returns','return'),('leaps','leap'),('crawls','crawl'),('floats','float'),
 ('glides','glide'),('wakes','wake'),('roams','roam'),('grazes','graze'),('naps','nap'),('barks','bark'),
 ('howls','howl'),('growls','growl'),('chirps','chirp'),('hops','hop'),('darts','dart'),('creeps','creep'),
 ('wades','wade'),('perches','perch'),('nests','nest'),('forages','forage'),('crouches','crouch'),
 ('pounces','pounce'),('scurries','scurry'),('burrows','burrow'),('paddles','paddle'),('drifts','drift'),
 ('circles','circle'),('soars','soar'),('dives','dive'),('waddles','waddle'),('stalks','stalk'),
 ('prowls','prowl'),('sniffs','sniff'),('licks','lick'),('scratches','scratch'),('shivers','shiver'),
 ('trembles','tremble'),('yawns','yawn'),('stretches','stretch'),('pauses','pause'),('lingers','linger'),
 ('settles','settle'),('stirs','stir'),
 ('hovers','hover'),('flaps','flap'),('flutters','flutter'),('swoops','swoop'),('plunges','plunge'),
 ('bounds','bound'),('gallops','gallop'),('trots','trot'),('canters','canter'),('ambles','amble'),
 ('strolls','stroll'),('saunters','saunter'),('shuffles','shuffle'),('slinks','slink'),('scampers','scamper'),
 ('scrambles','scramble'),('clambers','clamber'),('tumbles','tumble'),('rolls','roll'),('spins','spin'),
 ('twirls','twirl'),('sways','sway'),('rocks','rock'),('bobs','bob'),('nods','nod'),('blinks','blink'),
 ('snorts','snort'),('grunts','grunt'),('squeaks','squeak'),('squeals','squeal'),('hisses','hiss'),
 ('purrs','purr'),('mews','mew'),('bleats','bleat'),('clucks','cluck'),('quacks','quack'),('hoots','hoot'),
 ('croaks','croak'),('buzzes','buzz'),('chitters','chitter'),('whistles','whistle'),('whines','whine'),
 # --- +161 more ---
 ('walks','walk'),('marches','march'),('splashes','splash'),('wallows','wallow'),('lurks','lurk'),('loiters','loiter'),
 ('idles','idle'),('dawdles','dawdle'),('meanders','meander'),('zigzags','zigzag'),('weaves','weave'),('bolts','bolt'),
 ('sprints','sprint'),('scuttles','scuttle'),('scoots','scoot'),('skitters','skitter'),('skips','skip'),('bounces','bounce'),
 ('vaults','vault'),('springs','spring'),('lunges','lunge'),('charges','charge'),('rushes','rush'),('races','race'),
 ('dashes','dash'),('hurries','hurry'),('hastens','hasten'),('flees','flee'),('scatters','scatter'),('disperses','disperse'),
 ('retreats','retreat'),('advances','advance'),('approaches','approach'),('departs','depart'),('arrives','arrive'),('remains','remain'),
 ('stays','stay'),('paces','pace'),('strides','stride'),('stomps','stomp'),('stamps','stamp'),('tramps','tramp'),
 ('trudges','trudge'),('plods','plod'),('lumbers','lumber'),('slogs','slog'),('limps','limp'),('hobbles','hobble'),
 ('staggers','stagger'),('teeters','teeter'),('wobbles','wobble'),('totters','totter'),('reels','reel'),('lurches','lurch'),
 ('sags','sag'),('slumps','slump'),('slouches','slouch'),('leans','lean'),('roosts','roost'),('squats','squat'),
 ('kneels','kneel'),('sprawls','sprawl'),('reclines','recline'),('lounges','lounge'),('dozes','doze'),('slumbers','slumber'),
 ('dreams','dream'),('drowses','drowse'),('rouses','rouse'),('awakens','awaken'),('arises','arise'),('rises','rise'),
 ('descends','descend'),('ascends','ascend'),('mounts','mount'),('scales','scale'),('tunnels','tunnel'),('roots','root'),
 ('rummages','rummage'),('probes','probe'),('pokes','poke'),('nudges','nudge'),('nuzzles','nuzzle'),('paws','paw'),
 ('claws','claw'),('scrapes','scrape'),('gnaws','gnaw'),('nibbles','nibble'),('chews','chew'),('munches','munch'),
 ('browses','browse'),('pecks','peck'),('sips','sip'),('laps','lap'),('gulps','gulp'),('swallows','swallow'),
 ('drinks','drink'),('feasts','feast'),('scavenges','scavenge'),('hoards','hoard'),('stashes','stash'),('chases','chase'),
 ('pursues','pursue'),('tracks','track'),('trails','trail'),('ambushes','ambush'),('snatches','snatch'),('seizes','seize'),
 ('grabs','grab'),('clutches','clutch'),('grasps','grasp'),('clasps','clasp'),('grips','grip'),('tugs','tug'),
 ('hauls','haul'),('drags','drag'),('heaves','heave'),('shoves','shove'),('pushes','push'),('thrusts','thrust'),
 ('swats','swat'),('flails','flail'),('thrashes','thrash'),('writhes','writhe'),('squirms','squirm'),('wriggles','wriggle'),
 ('twists','twist'),('coils','coil'),('curls','curl'),('uncoils','uncoil'),('unfurls','unfurl'),('flexes','flex'),
 ('arches','arch'),('hunches','hunch'),('bristles','bristle'),('shudders','shudder'),('quivers','quiver'),('quakes','quake'),
 ('flinches','flinch'),('recoils','recoil'),('cringes','cringe'),('cowers','cower'),('hunkers','hunker'),('huddles','huddle'),
 ('nestles','nestle'),('snuggles','snuggle'),('cuddles','cuddle'),('preens','preen'),('grooms','groom'),('bathes','bathe'),
 ('surfaces','surface'),('coasts','coast'),('sails','sail'),('skims','skim'),('flits','flit'),('wheels','wheel'),
 ('banks','bank'),('veers','veer'),('swerves','swerve'),('spirals','spiral'),('plummets','plummet'),
]
VERBS = [{'sg':a,'pl':b} for a,b in dict.fromkeys(_VERB)]

COPULAS = [{'sg':'is','pl':'are'}]

_NOUN = [
 'duck','rabbit','frog','snake','lion','goat','bear','wolf','crab','hawk','toad','newt','mole','crow',
 'snail','moth','otter','badger','hare','vole','finch','wren','dove','swan','heron','robin','beetle',
 'spider','lizard','stoat','weasel','ferret','beaver','squirrel','gopher','turtle','gecko','iguana',
 'parrot','falcon','eagle','pigeon','quail','turkey','puffin','penguin','shrew','skunk','raccoon',
 'cricket','locust','hornet','aphid','firefly','ladybug','clam','oyster','prawn','shrimp','lobster',
 'eel','trout','carp','perch','pike','bass','cod','crane','stork','egret','plover','warbler','thrush',
 'raven','jay','cobra','viper','adder','boa','skink']
NOUNS = [{'sg':w, 'pl':(w+'es' if w.endswith(('x','s','sh','ch')) else w+'s')} for w in dict.fromkeys(_NOUN)]

QUANT = {'sg':['one','every','a single'], 'pl':['many','several','two']}

# --- extra banks for the depth-0 frames ---
ADJECTIVES = [
    'purple','tiny','clever','sleepy','angry','fuzzy','golden','spotted','silent','brave',
    'hungry','gentle','wild','striped','ancient','massive','small','large','little','big',
    'huge','giant','minor','major','vast','slight','broad','narrow','wide','thick',
    'thin','red','blue','green','yellow','orange','pink','brown','black','white',
    'gray','silver','bronze','crimson','scarlet','amber','fast','slow','quick','swift',
    'rapid','nimble','agile','sluggish','brisk','speedy','calm','quiet','loud','noisy',
    'fierce','tame','timid','bold','shy','meek','docile','feisty','happy','sad',
    'merry','gloomy','cheerful','glum','jolly','somber','joyful','weary','warm','cold',
    'cool','hot','chilly','frosty','icy','mild','balmy','tepid','soft','hard',
    'rough','smooth','bumpy','silky','furry','fluffy','coarse','sleek','bright','dark',
    'dim','dull','shiny','glossy','murky','pale','vivid','faded','young','old',
    'aged','youthful','elderly','mature','juvenile','strong','weak','sturdy','frail','robust',
    'feeble','mighty','puny','tough','fragile','clean','dirty','muddy','dusty','grimy',
    'spotless','filthy','tidy','messy','soiled','hairy','bald','scaly','feathered','woolly',
    'leathery','downy','bristly','shaggy','clumsy','graceful','elegant','awkward','lithe','ungainly',
    'dainty','curious','wary','alert','drowsy','restless','serene','placid','nervous','jittery',
    'edgy','lonely','social','friendly','hostile','savage','vicious','plump','skinny','lean',
    'stout','chubby','scrawny','portly','slender','lanky','burly','hidden','visible','secret',
    'obvious','sneaky','furtive','blatant','lazy','busy','idle','active','diligent','eager',
    'keen','foolish','wise','sharp','dense','astute','glowing','dewy','sandy','rocky',
    'leafy','mossy','grassy','weedy','crooked','straight','bent','curved','coiled','twisted',
    'gnarled','jagged','hollow','solid','airy','spongy','rigid','limp','stiff','supple',
    'loyal','fickle','steady','flighty','constant','erratic','dependable','wayward','mellow','harsh',
    'tender','brutal','kindly','cruel','humane','matted','tangled','groomed','ruffled','preened',
    'unkempt','wrinkled','creased','taut','saggy','firm','dotted','barred','banded','mottled',
    'dappled','flecked','freckled','velvety','satiny','rubbery','waxy','oily','greasy','fragrant',
    'musky','sour','sweet','bitter','pungent','acrid','earthy',
]
RC_PRONOUNS = ["he","she","they","I","we","you"]
RC_VERBS    = ["saw","chased","found","fed","held","raised","caught",
               "watched","named","loved","trained","kept","freed","led"]
SG_DEMS = ["this","that"]
PL_DEMS = ["these","those","some"]

ATTRACTORS = [0, 1, 2, 3]
N_FRAMES   = 44
PER_CELL   = 840


# ============================================================
#  Filter: keep only target words that are a SINGLE token
# ============================================================
def _tokenizer(lm):
    candidates = [
        getattr(lm, "tokenizer", None),
        getattr(getattr(lm, "processor", None), "tokenizer", None),
        getattr(getattr(lm, "tokenizer", None), "tokenizer", None),
    ]
    for c in candidates:
        if c is not None and hasattr(c, "encode"):
            return c
    raise AttributeError("Could not find a tokenizer with .encode on lm")

TOK = _tokenizer(lm)
print(type(TOK).__name__, "| has encode:", hasattr(TOK, "encode"))

def n_tokens(word, leading_space=True):
    s = (" " + word) if leading_space else word
    return len(TOK.encode(s, add_special_tokens=False))

def is_single_token(word):
    return n_tokens(word) == 1

verb_targets   = sorted({v[k] for v in VERBS    for k in ("sg", "pl")})
copula_targets = sorted({c[k] for c in COPULAS  for k in ("sg", "pl")})
noun_targets   = sorted({n[k] for n in NOUNS    for k in ("sg", "pl")})

def split_single(words):
    keep, drop = [], []
    for w in words:
        (keep if is_single_token(w) else drop).append((w, n_tokens(w)))
    return keep, drop

verb_keep,   verb_drop   = split_single(verb_targets)
cop_keep,    cop_drop    = split_single(copula_targets)
noun_keep,   noun_drop   = split_single(noun_targets)

print("=== DROPPED (multi-token) target words ===")
for label, drop in [("VERB", verb_drop), ("COPULA", cop_drop), ("NOUN", noun_drop)]:
    if drop:
        for w, nt in drop:
            print(f"  [{label}] {w!r}  -> {nt} tokens")
    else:
        print(f"  [{label}] none")

def filter_pairs(bank):
    kept, removed = [], []
    for d in bank:
        if is_single_token(d["sg"]) and is_single_token(d["pl"]):
            kept.append(d)
        else:
            removed.append(d)
    return kept, removed

VERBS_f,   verbs_rm   = filter_pairs(VERBS)
COPULAS_f, cop_rm     = filter_pairs(COPULAS)
NOUNS_f,   nouns_rm   = filter_pairs(NOUNS)

print("\n=== DROPPED minimal-pair entries (one or both forms multi-token) ===")
for label, rm in [("VERB", verbs_rm), ("COPULA", cop_rm), ("NOUN", nouns_rm)]:
    if rm:
        for d in rm:
            why = []
            if not is_single_token(d["sg"]): why.append(f"sg={d['sg']!r}({n_tokens(d['sg'])})")
            if not is_single_token(d["pl"]): why.append(f"pl={d['pl']!r}({n_tokens(d['pl'])})")
            print(f"  [{label}] {d}  ->  {', '.join(why)}")
    else:
        print(f"  [{label}] none")

VERBS, COPULAS, NOUNS = VERBS_f, COPULAS_f, NOUNS_f
print(f"\nKept: {len(VERBS)} verbs, {len(COPULAS)} copulas, {len(NOUNS)} nouns")


def pp_chain(offset, n, dist_num):
    return " ".join(
        f"{PREPS[(offset+j) % len(PREPS)]} {DISTRACTORS[(offset+j) % len(DISTRACTORS)][dist_num]}"
        for j in range(n)
    )
def cap(s): return s[0].upper() + s[1:]
def cond(t,num,n,dn): return f"target_{t}_num_{num}_att{n}_{dn}"

def mk(*parts):
    """Join non-empty parts, collapse spaces, capitalize. NO trailing period."""
    s = re.sub(r"\s+", " ", " ".join(p for p in parts if p)).strip()
    return cap(s)

rows = []
for si, subj in enumerate(SUBJECTS):
    for n in ATTRACTORS:
        for num in ('sg','pl'):
            other = 'pl' if num == 'sg' else 'sg'
            dn = other
            for f in range(N_FRAMES):
                offset = si + f

                if n == 0:
                    # ---------- DEPTH-0 ----------
                    # VERB: adjective-modified -> "The purple bears run"
                    adj = ADJECTIVES[(si + f) % len(ADJECTIVES)]
                    v = VERBS[(si + f) % len(VERBS)]
                    rows.append({
                        'target_type':'verb','attractors':0,'target_num':num,
                        'condition':cond('verb',num,0,dn),
                        'subject_word': subj[num], 'target_word': v[num],
                        'base_sentence': mk("The", adj, subj[num]),
                        'good': mk("The", adj, subj[num], v[num]),
                        'bad':  mk("The", adj, subj[num], v[other]),
                    })

                    # COPULA: adjective-modified -> "The purple bears are"
                    adj_c = ADJECTIVES[(si + f + 7) % len(ADJECTIVES)]
                    c = COPULAS[0]
                    rows.append({
                        'target_type':'copula','attractors':0,'target_num':num,
                        'condition':cond('copula',num,0,dn),
                        'subject_word': subj[num], 'target_word': c[num],
                        'base_sentence': mk("The", adj_c, subj[num]),
                        'good': mk("The", adj_c, subj[num], c[num]),
                        'bad':  mk("The", adj_c, subj[num], c[other]),
                    })

                    # NOUN: demonstrative-controlled -> "He chased some bears"
                    tn  = NOUNS[(si + f) % len(NOUNS)]
                    pron = RC_PRONOUNS[(si + f) % len(RC_PRONOUNS)]
                    rcv  = RC_VERBS[(si + f) % len(RC_VERBS)]
                    sg_dem = SG_DEMS[f % len(SG_DEMS)]
                    pl_dem = PL_DEMS[f % len(PL_DEMS)]
                    dem     = sg_dem if num == 'sg' else pl_dem
                    bad_dem = pl_dem if num == 'sg' else sg_dem
                    rows.append({
                        'target_type':'noun','attractors':0,'target_num':num,
                        'condition':cond('noun',num,0,dn),
                        'subject_word': tn[num], 'target_word': tn[num],
                        'base_sentence': mk(cap(pron), rcv, dem),
                        'good': mk(cap(pron), rcv, dem, tn[num]),
                        'bad':  mk(cap(pron), rcv, bad_dem, tn[other]),
                    })
                    continue

                # ---------- DEPTH >=1: PP-attractor frames ----------
                pp = pp_chain(offset, n, dn)

                # verb target
                v = VERBS[(si + f) % len(VERBS)]
                rows.append({
                    'target_type':'verb','attractors':n,'target_num':num,
                    'condition':cond('verb',num,n,dn),
                    'subject_word': subj[num], 'target_word': v[num],
                    'base_sentence': mk("The", subj[num], pp),
                    'good': mk("The", subj[num], pp, v[num]),
                    'bad':  mk("The", subj[num], pp, v[other]),
                })

                # copula target
                c = COPULAS[0]
                rows.append({
                    'target_type':'copula','attractors':n,'target_num':num,
                    'condition':cond('copula',num,n,dn),
                    'subject_word': subj[num], 'target_word': c[num],
                    'base_sentence': mk("The", subj[num], pp),
                    'good': mk("The", subj[num], pp, c[num]),
                    'bad':  mk("The", subj[num], pp, c[other]),
                })

                # noun target (quantifier sets number, plural host keeps verb fixed)
                tn = NOUNS[(si + f) % len(NOUNS)]
                q  = QUANT[num][f % len(QUANT[num])]
                rows.append({
                    'target_type':'noun','attractors':n,'target_num':num,
                    'condition':cond('noun',num,n,dn),
                    'subject_word': tn[num], 'target_word': tn[num],
                    'base_sentence': mk("The", subj['pl'], pp, "hold", q),
                    'good': mk("The", subj['pl'], pp, "hold", q, tn[num]),
                    'bad':  mk("The", subj['pl'], pp, "hold", q, tn[other]),
                })

df = pd.DataFrame(rows).drop_duplicates(subset=['good']).reset_index(drop=True)
print("raw per condition (unique base sentences):")
print(df.groupby('target_type')['base_sentence'].nunique(), "\n")

# --- keep only sentences whose 'good' tokenizes to the same length, per (condition x attractors) ---
def tok_len(text):
    return len(TOK.encode(text, add_special_tokens=False))

df['tok_len'] = df['good'].map(tok_len)

dropped_uneven = 0
keep_masks = []
for ttype in df['target_type'].unique():
    for n in ATTRACTORS:
        cell_mask = (df['target_type'] == ttype) & (df['attractors'] == n)
        cell = df[cell_mask]
        if len(cell) == 0:
            continue
        mode_len = cell['tok_len'].mode().iloc[0]
        keep = cell_mask & (df['tok_len'] == mode_len)
        dropped_uneven += int(cell_mask.sum() - keep.sum())
        keep_masks.append(keep)

uniform = pd.concat([df[m] for m in keep_masks], ignore_index=False)
print(f"dropped {dropped_uneven} rows with off-length tokenization\n")

pieces = []
for ttype in uniform['target_type'].unique():
    for n in ATTRACTORS:
        cell = uniform[(uniform['target_type'] == ttype) & (uniform['attractors'] == n)]
        pieces.append(cell.sample(min(len(cell), PER_CELL), random_state=0))

bal = pd.concat(pieces, ignore_index=True)
bal.insert(0, 'idx', range(len(bal)))

print(bal.pivot_table(index='target_type', columns='attractors',
                      values='idx', aggfunc='count', fill_value=0))
print("\nper condition (unique base sentences):")
print(bal.groupby('target_type')['base_sentence'].nunique())
print("TOTAL rows:", len(bal), "| TOTAL unique base sentences:", bal['base_sentence'].nunique())

bal.to_csv("agreement_target_grid.csv", index=False)

Qwen2Tokenizer | has encode: True
=== DROPPED (multi-token) target words ===
  [VERB] 'amble'  -> 2 tokens
  [VERB] 'ambles'  -> 2 tokens
  [VERB] 'ambushes'  -> 2 tokens
  [VERB] 'arches'  -> 2 tokens
  [VERB] 'ascends'  -> 2 tokens
  [VERB] 'awakens'  -> 2 tokens
  [VERB] 'barks'  -> 2 tokens
  [VERB] 'bathe'  -> 2 tokens
  [VERB] 'bathes'  -> 2 tokens
  [VERB] 'bleat'  -> 2 tokens
  [VERB] 'bleats'  -> 2 tokens
  [VERB] 'blinks'  -> 2 tokens
  [VERB] 'bobs'  -> 2 tokens
  [VERB] 'bounces'  -> 2 tokens
  [VERB] 'bristle'  -> 2 tokens
  [VERB] 'bristles'  -> 2 tokens
  [VERB] 'browses'  -> 2 tokens
  [VERB] 'burrow'  -> 2 tokens
  [VERB] 'burrows'  -> 2 tokens
  [VERB] 'buzzes'  -> 2 tokens
  [VERB] 'canter'  -> 2 tokens
  [VERB] 'canters'  -> 2 tokens
  [VERB] 'chases'  -> 2 tokens
  [VERB] 'chews'  -> 2 tokens
  [VERB] 'chirp'  -> 2 tokens
  [VERB] 'chirps'  -> 2 tokens
  [VERB] 'chitter'  -> 2 tokens
  [VERB] 'chitters'  -> 2 tokens
  [VERB] 'clamber'  -> 2 tokens
  [VERB] 'clamber

In [ ]:
# ============================================================
#  Wug-subject version: same frames as main block, subject animal -> [wug]/[wugs].
#  Depth-0: verb & copula are adjective-modified, noun is demonstrative-controlled.
#  Depth>=1: PP attractors. SUBJECTS only drives the attractor-chain offset.
#  Run AFTER the main block (reuses ADJECTIVES, RC_PRONOUNS, RC_VERBS,
#  SG_DEMS, PL_DEMS, VERBS, COPULAS, NOUNS, QUANT, helpers, tok_len).
# ============================================================
WUG = {'sg': '[wug]', 'pl': '[wugs]'}

rows_wug = []
for si, subj in enumerate(SUBJECTS):
    for n in ATTRACTORS:
        for num in ('sg', 'pl'):
            other = 'pl' if num == 'sg' else 'sg'
            dn = other
            for f in range(N_FRAMES):
                offset = si + f

                if n == 0:
                    # ---------- DEPTH-0 ----------
                    # VERB: adjective-modified -> "The purple [wugs] jump"
                    adj = ADJECTIVES[(si + f) % len(ADJECTIVES)]
                    v = VERBS[(si + f) % len(VERBS)]
                    rows_wug.append({
                        'target_type':'verb','attractors':0,'target_num':num,
                        'condition':cond('verb',num,0,dn),
                        'subject_word': WUG[num], 'target_word': v[num],
                        'base_sentence': mk("The", adj, WUG[num]),
                        'good': mk("The", adj, WUG[num], v[num]),
                        'bad':  mk("The", adj, WUG[num], v[other]),
                    })

                    # COPULA: adjective-modified -> "The purple [wugs] are"
                    adj_c = ADJECTIVES[(si + f + 7) % len(ADJECTIVES)]
                    c = COPULAS[0]
                    rows_wug.append({
                        'target_type':'copula','attractors':0,'target_num':num,
                        'condition':cond('copula',num,0,dn),
                        'subject_word': WUG[num], 'target_word': c[num],
                        'base_sentence': mk("The", adj_c, WUG[num]),
                        'good': mk("The", adj_c, WUG[num], c[num]),
                        'bad':  mk("The", adj_c, WUG[num], c[other]),
                    })

                    # NOUN: demonstrative-controlled -> "He chased some [wugs]" (target = wug)
                    pron = RC_PRONOUNS[(si + f) % len(RC_PRONOUNS)]
                    rcv  = RC_VERBS[(si + f) % len(RC_VERBS)]
                    sg_dem = SG_DEMS[f % len(SG_DEMS)]
                    pl_dem = PL_DEMS[f % len(PL_DEMS)]
                    dem     = sg_dem if num == 'sg' else pl_dem
                    bad_dem = pl_dem if num == 'sg' else sg_dem
                    rows_wug.append({
                        'target_type':'noun','attractors':0,'target_num':num,
                        'condition':cond('noun',num,0,dn),
                        'subject_word': WUG[num], 'target_word': WUG[num],
                        'base_sentence': mk(cap(pron), rcv, dem),
                        'good': mk(cap(pron), rcv, dem, WUG[num]),
                        'bad':  mk(cap(pron), rcv, bad_dem, WUG[other]),
                    })
                    continue

                # ---------- DEPTH >=1: PP-attractor frames ----------
                pp = pp_chain(offset, n, dn)

                # verb target — wug subject is the controller
                v = VERBS[(si + f) % len(VERBS)]
                rows_wug.append({
                    'target_type':'verb','attractors':n,'target_num':num,
                    'condition':cond('verb',num,n,dn),
                    'subject_word': WUG[num], 'target_word': v[num],
                    'base_sentence': mk("The", WUG[num], pp),
                    'good': mk("The", WUG[num], pp, v[num]),
                    'bad':  mk("The", WUG[num], pp, v[other]),
                })

                # copula target — wug subject is the controller
                c = COPULAS[0]
                rows_wug.append({
                    'target_type':'copula','attractors':n,'target_num':num,
                    'condition':cond('copula',num,n,dn),
                    'subject_word': WUG[num], 'target_word': c[num],
                    'base_sentence': mk("The", WUG[num], pp),
                    'good': mk("The", WUG[num], pp, c[num]),
                    'bad':  mk("The", WUG[num], pp, c[other]),
                })

                # noun target — target noun is the wug; host subject kept plural so 'hold' is fixed
                q  = QUANT[num][f % len(QUANT[num])]
                rows_wug.append({
                    'target_type':'noun','attractors':n,'target_num':num,
                    'condition':cond('noun',num,n,dn),
                    'subject_word': WUG[num], 'target_word': WUG[num],
                    'base_sentence': mk("The", WUG['pl'], pp, "hold", q),
                    'good': mk("The", WUG['pl'], pp, "hold", q, WUG[num]),
                    'bad':  mk("The", WUG['pl'], pp, "hold", q, WUG[other]),
                })

df_wug = pd.DataFrame(rows_wug).drop_duplicates(subset=['good']).reset_index(drop=True)
print("raw per condition (wug, unique base sentences):")
print(df_wug.groupby('target_type')['base_sentence'].nunique(), "\n")

# same uniform-token-length filter per (condition x attractors)
df_wug['tok_len'] = df_wug['good'].map(tok_len)
dropped_uneven = 0
keep_masks = []
for ttype in df_wug['target_type'].unique():
    for n in ATTRACTORS:
        cell_mask = (df_wug['target_type'] == ttype) & (df_wug['attractors'] == n)
        cell = df_wug[cell_mask]
        if len(cell) == 0:
            continue
        mode_len = cell['tok_len'].mode().iloc[0]
        keep = cell_mask & (df_wug['tok_len'] == mode_len)
        dropped_uneven += int(cell_mask.sum() - keep.sum())
        keep_masks.append(keep)
uniform_wug = pd.concat([df_wug[m] for m in keep_masks], ignore_index=False)
print(f"dropped {dropped_uneven} rows with off-length tokenization (wug)\n")

pieces = []
for ttype in uniform_wug['target_type'].unique():
    for n in ATTRACTORS:
        cell = uniform_wug[(uniform_wug['target_type'] == ttype) & (uniform_wug['attractors'] == n)]
        pieces.append(cell.sample(min(len(cell), PER_CELL), random_state=0))

bal_wug = pd.concat(pieces, ignore_index=True)
bal_wug.insert(0, 'idx', range(len(bal_wug)))

print(bal_wug.pivot_table(index='target_type', columns='attractors',
                          values='idx', aggfunc='count', fill_value=0))
print("\nper condition (wug, unique base sentences):")
print(bal_wug.groupby('target_type')['base_sentence'].nunique())
print("TOTAL rows:", len(bal_wug), "| TOTAL unique base sentences:", bal_wug['base_sentence'].nunique())

bal_wug.to_csv("agreement_target_grid_wug.csv", index=False)

raw per condition (wug):
target_type
copula    1674
noun      4944
verb      2120
dtype: int64 

dropped 4085 rows with off-length tokenization (wug)

attractors     0    1    2    3
target_type                    
copula        84  332  266  229
noun         210  840  703  614
verb         530  332  266  229

per condition (wug):
target_type
copula     911
noun      2367
verb      1357
dtype: int64
TOTAL: 4635
